# Phase 12 — Repository Hygiene and Notebook Runtime Bootstrap

This notebook keeps training execution notebook-first. It must run from a clean kernel without relying on local virtual environments, hidden state, or Python package entrypoints under `training/*.py`.

Durable outputs from this notebook should be written to `reports/` or another explicitly documented artifact path.


## Step 12.1 — Virtual environment hygiene

### Purpose
Remove local runtime noise from notebook source so training can be reviewed and reproduced without committed virtual environments.

### Required input
Repository file tree, `training/.gitignore`, root `.gitignore`, and any directories matching `training/notebooks/**/venv` or `training/notebooks/**/.venv`.

### Action
Identify notebook-local virtual environments, keep ignore rules for future local environments, and verify no tracked training source depends on notebook-local virtual environment directories. Local virtual environments may exist on a workstation, but they must stay ignored and must not be required by any notebook.

### Expected output
Notebook source remains lightweight. Local environments are ignored, not required for model training evidence.

### Verification
A clean tree check shows no tracked training artifact under `training/notebooks/**/venv` or `training/notebooks/**/.venv`.


In [21]:
from pathlib import Path
import subprocess


def find_repo_root(start=None) -> Path:
    """Resolve the repository root from a notebook or repository working directory."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "GAP_MODEL_TRAINING.md").exists() and (candidate / "training" / "notebooks").exists():
            return candidate
    raise RuntimeError("Repository root not found. Start notebook inside bisakerja-model repository.")


REPO_ROOT = find_repo_root()
TRAINING_ROOT = REPO_ROOT / "training"
NOTEBOOK_ROOT = TRAINING_ROOT / "notebooks"
REPORTS_DIR = REPO_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

venv_dirs = sorted(
    [p for pattern in ("**/venv", "**/.venv") for p in NOTEBOOK_ROOT.glob(pattern) if p.is_dir()]
)

def git_ls_files(*patterns: str) -> list[str]:
    result = subprocess.run(
        ["git", "ls-files", *patterns],
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
        check=True,
    )
    return [line for line in result.stdout.splitlines() if line.strip()]

tracked_venv_files = git_ls_files(
    "training/notebooks/**/venv/**",
    "training/notebooks/**/.venv/**",
)

root_gitignore = (REPO_ROOT / ".gitignore").read_text(encoding="utf-8") if (REPO_ROOT / ".gitignore").exists() else ""
training_gitignore = (TRAINING_ROOT / ".gitignore").read_text(encoding="utf-8") if (TRAINING_ROOT / ".gitignore").exists() else ""
required_ignore_rules = {
    ".gitignore": ["training/notebooks/**/venv/", "training/notebooks/**/.venv/"],
    "training/.gitignore": ["notebooks/**/venv/", "notebooks/**/.venv/"],
}
missing_ignore_rules = []
for rule in required_ignore_rules[".gitignore"]:
    if rule not in root_gitignore:
        missing_ignore_rules.append(f".gitignore:{rule}")
for rule in required_ignore_rules["training/.gitignore"]:
    if rule not in training_gitignore:
        missing_ignore_rules.append(f"training/.gitignore:{rule}")

assert not tracked_venv_files, f"Tracked notebook virtual environment files found: {tracked_venv_files[:10]}"
assert not missing_ignore_rules, f"Missing virtual environment ignore rules: {missing_ignore_rules}"

venv_hygiene = {
    "local_venv_directories_found": [str(path.relative_to(REPO_ROOT)) for path in venv_dirs],
    "tracked_venv_file_count": len(tracked_venv_files),
    "ignore_rules_verified": True,
    "policy": "Notebook-local virtual environments are ignored local runtime state and are not required training artifacts.",
}
venv_hygiene


{'local_venv_directories_found': ['training/notebooks/venv',
  'training/notebooks/venv/lib/python3.14/site-packages/jedi/third_party/typeshed/stdlib/venv'],
 'tracked_venv_file_count': 0,
 'ignore_rules_verified': True,
 'policy': 'Notebook-local virtual environments are ignored local runtime state and are not required training artifacts.'}

## Step 12.2 — Notebook runtime setup

### Purpose
Define reproducible runtime setup inside this notebook without extracting training logic into Python package files.

### Required input
Repository root, Python runtime, notebook kernel, dependency list, and deterministic seed value.

### Action
Add setup cells for repo path resolution, dependency checks, deterministic seeds, logging, and artifact hashing.

### Expected output
Every later notebook cell can resolve repo paths, use fixed seeds, and hash artifacts consistently from a clean kernel.

### Verification
Restart kernel and run all cells top-to-bottom. Runtime setup must not import `training.train`, `training.evaluate`, or other package entrypoints.


In [22]:
import hashlib
import importlib.util
import json
import logging
import os
import platform
import random
import sys
from datetime import datetime, timezone
from typing import Any

PHASE_ID = "phase_12_repository_hygiene_runtime_bootstrap"
SEED = 202612
CURRENT_COMPLETED_PHASE = 19
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)

try:
    import numpy as np  # optional acceleration dependency for later phases
except Exception:  # pragma: no cover - notebook environment dependent
    np = None
else:
    np.random.seed(SEED)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(PHASE_ID)
logger.info("Repository root resolved to %s", REPO_ROOT)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def file_record(path: Path) -> dict[str, Any]:
    resolved = (REPO_ROOT / path).resolve() if not path.is_absolute() else path.resolve()
    exists = resolved.exists()
    return {
        "path": str(resolved.relative_to(REPO_ROOT)) if exists and resolved.is_relative_to(REPO_ROOT) else str(path),
        "exists": exists,
        "size_bytes": resolved.stat().st_size if exists and resolved.is_file() else None,
        "sha256": sha256_file(resolved) if exists and resolved.is_file() else None,
    }

required_runtime_dependencies = ["json", "hashlib", "pathlib", "logging", "random"]
phase_17_plus_dependencies = ["numpy", "pandas", "sklearn", "sentence_transformers"]
optional_compatibility_dependencies = ["tensorflow"]
all_dependency_names = [
    *required_runtime_dependencies,
    *phase_17_plus_dependencies,
    *optional_compatibility_dependencies,
]
dependency_status = {name: importlib.util.find_spec(name) is not None for name in all_dependency_names}
missing_required_dependencies = [name for name in required_runtime_dependencies if not dependency_status[name]]
missing_phase_17_plus_dependencies = [name for name in phase_17_plus_dependencies if not dependency_status[name]]
assert not missing_required_dependencies, f"Missing required Phase 12 runtime dependencies: {missing_required_dependencies}"

runtime_readiness = {
    "phase_12_base_ready": not missing_required_dependencies,
    "phase_17_plus_training_ready": not missing_phase_17_plus_dependencies,
    "legacy_tensorflow_model_load_available": bool(dependency_status.get("tensorflow")),
    "missing_phase_17_plus_dependencies": missing_phase_17_plus_dependencies,
    "policy": {
        "phase_12_required": required_runtime_dependencies,
        "phase_17_plus_required_for_e5_training": phase_17_plus_dependencies,
        "legacy_model_load_optional": optional_compatibility_dependencies,
    },
}

runtime_warnings = []
if missing_phase_17_plus_dependencies:
    runtime_warnings.append(
        "Phase 12 can run, but Phase 17+ E5 training readiness is blocked until these dependencies exist: "
        + ", ".join(missing_phase_17_plus_dependencies)
    )
if not dependency_status.get("tensorflow"):
    runtime_warnings.append("TensorFlow unavailable; legacy Keras model load compatibility is skipped in this kernel.")

runtime_snapshot = {
    "phase_id": PHASE_ID,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "repo_root": str(REPO_ROOT),
    "seed": SEED,
    "dependency_status": dependency_status,
    "runtime_readiness": runtime_readiness,
    "runtime_warnings": runtime_warnings,
}
runtime_snapshot


2026-06-02 11:33:16,398 INFO Repository root resolved to /Users/macbookpro/Development/bisakerja-model


{'phase_id': 'phase_12_repository_hygiene_runtime_bootstrap',
 'generated_at': '2026-06-02T04:33:16.401019+00:00',
 'python_version': '3.14.2 (main, Dec  5 2025, 16:49:16) [Clang 16.0.0 (clang-1600.0.26.6)]',
 'python_executable': '/Users/macbookpro/Development/bisakerja-model/training/notebooks/venv/bin/python',
 'platform': 'macOS-14.8.2-arm64-arm-64bit-Mach-O',
 'repo_root': '/Users/macbookpro/Development/bisakerja-model',
 'seed': 202612,
 'dependency_status': {'json': True,
  'hashlib': True,
  'pathlib': True,
  'logging': True,
  'random': True,
  'numpy': True,
  'pandas': True,
  'sklearn': True,
  'sentence_transformers': True,
  'tensorflow': False},
 'runtime_readiness': {'phase_12_base_ready': True,
  'phase_17_plus_training_ready': True,
  'legacy_tensorflow_model_load_available': False,
  'missing_phase_17_plus_dependencies': [],
  'policy': {'phase_12_required': ['json',
    'hashlib',
    'pathlib',
    'logging',
    'random'],
   'phase_17_plus_required_for_e5_traini

## Step 12.3 — Versioned notebook config blocks

### Purpose
Keep model configuration explicit and versioned while preserving notebook-first execution.

### Required input
Model scopes `jobfit_v2`, `ats_quality_v1`, shared feature/schema settings, label versions, split seed, and artifact paths.

### Action
Define config dictionaries in notebook cells and export normalized JSON snapshots to `reports/`.

### Expected output
Each promoted config includes schema version, model version, label version, split seed, and artifact paths.

### Verification
Config validation cells fail when required keys are missing or source artifact paths are invalid.


In [23]:
CONFIG_SCHEMA_VERSION = "notebook-config-v1"
DEFAULT_SPLIT_SEED = SEED

notebook_configs: dict[str, dict[str, Any]] = {
    "shared_features_schema_v1": {
        "schema_version": CONFIG_SCHEMA_VERSION,
        "model_version": "shared-feature-schema-v1",
        "label_version": "shared-label-boundary-v1",
        "split_seed": DEFAULT_SPLIT_SEED,
        "language_values": ["ID", "EN", "MIXED", "UNKNOWN"],
        "score_range": {"min": 0, "max": 100},
        "model_owned_outputs": ["jobFitAlignment", "atsFriendliness", "overallImpression"],
        "wrapper_owned_outputs": ["topActionables", "sectionReviews", "hydratedJobDetails"],
        "source_artifact_paths": {
            "legacy_jobs": "legacy/dataset/indotech_job_cleaned.csv",
            "legacy_profiles": "legacy/dataset/techtalent_profile_cleaned.csv",
            "current_pairs_v2": "artifacts/pairs_v2.parquet",
            "legacy_model_card": "legacy/reports/model_card.json",
            "openapi_contract": "references/docs/generated/openapi.json",
        },
        "output_artifact_paths": {
            "config_report": "reports/phase_12_notebook_configs.json",
            "runtime_report": "reports/phase_12_repository_hygiene_runtime_bootstrap.json",
        },
    },
    "jobfit_v2": {
        "schema_version": CONFIG_SCHEMA_VERSION,
        "model_version": "jobfit-v2-e5-notebook-bootstrap",
        "label_version": "weak-label-balanced-v2",
        "split_seed": DEFAULT_SPLIT_SEED,
        "embedding_contract": {
            "embedding_model": "intfloat/e5-base-v2",
            "profile_cv_prefix": "query:",
            "job_prefix": "passage:",
            "normalized_embeddings": True,
            "fallback_policy": "TF-IDF or local-hash fallback is local-plumbing-only and must fail staging/production readiness gates.",
        },
        "approved_features": [
            "e5_cosine",
            "skill_overlap",
            "requirement_coverage",
            "role_match",
            "experience_match",
            "experience_gap_years",
            "language",
        ],
        "source_artifact_paths": {
            "pairs_v2": "artifacts/pairs_v2.parquet",
            "phase_17_embedding_manifest": "reports/phase_17_embedding_manifest.json",
            "phase_18_embedding_manifest": "reports/phase_18_embedding_manifest.json",
            "phase_18_model_metrics": "reports/phase_18_model_metrics.json",
            "phase_16_human_labels": "artifacts/manual_validation/phase_16_human_labels_frozen.csv",
        },
        "legacy_reference_artifact_paths": {
            "legacy_pairs": "legacy/artifacts/pairs.parquet",
            "legacy_job_embeddings": "legacy/cache/all_job_embeddings.npy",
            "legacy_job_index": "legacy/artifacts/job_index.json",
            "legacy_model": "legacy/models/model_jobfit_v1.keras",
        },
        "output_artifact_paths": {
            "pairs_v2_dataset": "artifacts/pairs_v2.parquet",
            "pair_generation_report": "reports/phase_15_balanced_pair_generation_splits.json",
            "pair_leakage_report": "reports/phase_15_leakage_report.json",
            "pair_distribution_diagnostics": "reports/phase_15_pair_distribution_diagnostics.json",
            "baseline_v2": "reports/phase_17_baseline_evaluation_v2.json",
            "model_card": "reports/model_card_jobfit_v2.json",
            "artifact_manifest": "reports/artifact_manifest_jobfit_v2.json",
        },
    },
    "ats_quality_v1": {
        "schema_version": CONFIG_SCHEMA_VERSION,
        "model_version": "ats-quality-v1-notebook-bootstrap",
        "label_version": "ats-quality-label-v1",
        "split_seed": DEFAULT_SPLIT_SEED,
        "issue_taxonomy": [
            "parseability",
            "section_completeness",
            "contact_detection",
            "date_detection",
            "metric_evidence",
            "formatting_risk",
            "empty_parse_risk",
        ],
        "source_artifact_paths": {
            "cv_benchmark_manifest": "reports/phase_19_cv_benchmark_manifest.json",
            "ats_issue_labels": "reports/phase_19_ats_issue_labels.json",
            "ats_scorer_metrics": "reports/phase_19_ats_scorer_metrics.json",
        },
        "output_artifact_paths": {
            "ats_quality_report": "reports/phase_19_ats_friendliness_benchmark_scorer.json",
            "model_card": "reports/model_card_ats_quality_v1.json",
            "artifact_manifest": "reports/artifact_manifest_ats_quality_v1.json",
        },
        "allow_missing_source_artifacts_until_phase": 19,
    },
}

required_config_keys = {"schema_version", "model_version", "label_version", "split_seed", "source_artifact_paths", "output_artifact_paths"}
config_errors: list[str] = []
for config_name, config in notebook_configs.items():
    missing_keys = sorted(required_config_keys - set(config))
    if missing_keys:
        config_errors.append(f"{config_name}: missing keys {missing_keys}")
    if not isinstance(config.get("split_seed"), int):
        config_errors.append(f"{config_name}: split_seed must be an int")
    for output_path in config.get("output_artifact_paths", {}).values():
        output_parent = (REPO_ROOT / output_path).parent
        if not output_parent.exists():
            config_errors.append(f"{config_name}: output parent does not exist: {output_parent.relative_to(REPO_ROOT)}")

for config_name, config in notebook_configs.items():
    allow_missing_until = config.get("allow_missing_source_artifacts_until_phase")
    for label, source_path in config.get("source_artifact_paths", {}).items():
        full_path = REPO_ROOT / source_path
        missing_allowed = allow_missing_until is not None and CURRENT_COMPLETED_PHASE < int(allow_missing_until)
        if not full_path.exists() and not missing_allowed:
            config_errors.append(f"{config_name}: missing source artifact {label}: {source_path}")
    for label, source_path in config.get("legacy_reference_artifact_paths", {}).items():
        full_path = REPO_ROOT / source_path
        if not full_path.exists():
            config_errors.append(f"{config_name}: missing legacy reference artifact {label}: {source_path}")

embedding_contract = notebook_configs["jobfit_v2"]["embedding_contract"]
if embedding_contract["embedding_model"] != "intfloat/e5-base-v2":
    config_errors.append("jobfit_v2: embedding model must be intfloat/e5-base-v2")
if embedding_contract["profile_cv_prefix"] != "query:" or embedding_contract["job_prefix"] != "passage:":
    config_errors.append("jobfit_v2: E5 prefixes must be query:/passage:")
if embedding_contract["normalized_embeddings"] is not True:
    config_errors.append("jobfit_v2: normalized_embeddings must be true")

assert not config_errors, "Config validation failed:" + "\n" + "\n".join(config_errors)

config_report_path = REPORTS_DIR / "phase_12_notebook_configs.json"
config_report_path.write_text(json.dumps(notebook_configs, indent=2, sort_keys=True), encoding="utf-8")
config_report = file_record(config_report_path)
config_report


{'path': 'reports/phase_12_notebook_configs.json',
 'exists': True,
 'size_bytes': 4069,
 'sha256': 'e8901a4dee6938cea0462eb2260cd94e06676f119d921b280b7f1c852b242b62'}

## Step 12.4 — Notebook verification cells

### Purpose
Make notebook readiness measurable without relying on hidden kernel state or script entrypoints.

### Required input
Required notebook list, source artifacts, config blocks, dependency names, and optional legacy model artifact path.

### Action
Add verification cells for required files, dependency availability, config completeness, artifact hashes, and model-load compatibility when TensorFlow is installed.

### Expected output
A verification summary can be displayed in the notebook and exported to `reports/phase_12_repository_hygiene_runtime_bootstrap.json`.

### Verification
Intentionally remove or rename a required input in a throwaway copy and confirm verification fails with a clear message.


In [24]:
required_notebooks = [
    "training/notebooks/phase_00_reproducibility_snapshot.ipynb",
    "training/notebooks/phase_01_data_audit_contracts.ipynb",
    "training/notebooks/phase_02_label_schema_baselines.ipynb",
    "training/notebooks/phase_03_normalization_feature_design.ipynb",
    "training/notebooks/phase_04_pair_generation_splits.ipynb",
    "training/notebooks/phase_05_baseline_evaluation.ipynb",
    "training/notebooks/phase_06_jobfit_training_experiments.ipynb",
    "training/notebooks/phase_07_ats_friendliness_scoring.ipynb",
    "training/notebooks/phase_08_overall_impression_signals.ipynb",
    "training/notebooks/phase_09_candidate_reranking.ipynb",
    "training/notebooks/phase_10_calibration_model_card.ipynb",
    "training/notebooks/phase_11_final_gate_review.ipynb",
    "training/notebooks/phase_12_repository_hygiene_runtime_bootstrap.ipynb",
    "training/notebooks/phase_13_data_snapshot_contract_freezing.ipynb",
    "training/notebooks/phase_14_normalization_feature_builder.ipynb",
    "training/notebooks/phase_15_balanced_pair_generation_splits.ipynb",
    "training/notebooks/phase_16_human_validation_label_governance.ipynb",
    "training/notebooks/phase_17_baseline_evaluation_v2.ipynb",
    "training/notebooks/phase_18_jobfit_training_v2.ipynb",
    "training/notebooks/phase_19_ats_friendliness_benchmark_scorer.ipynb",
    "training/notebooks/phase_19_5_jobfit_blocker_remediation.ipynb",
]
planned_notebooks = [
    "training/notebooks/phase_20_overall_impression_signals.ipynb",
    "training/notebooks/phase_21_backend_candidate_reranking.ipynb",
    "training/notebooks/phase_22_calibration_model_card_export.ipynb",
    "training/notebooks/phase_23_model_api_contract_validation.ipynb",
    "training/notebooks/phase_24_reproducibility_final_gate.ipynb",
]

required_files = [
    "GAP_MODEL_TRAINING.md",
    "GAP_MODEL_TRAINING.md",
    "training/README.md",
    "training/notebooks/README.md",
    ".gitignore",
    "training/.gitignore",
    *required_notebooks,
]

file_checks = {path: file_record(Path(path)) for path in required_files}
missing_required_files = [path for path, record in file_checks.items() if not record["exists"]]
assert not missing_required_files, f"Missing required Phase 12 files: {missing_required_files}"

planned_file_checks = {path: file_record(Path(path)) for path in planned_notebooks}

source_artifact_records = {}
for config_name, config in notebook_configs.items():
    for label, source_path in config.get("source_artifact_paths", {}).items():
        key = f"{config_name}.{label}"
        source_artifact_records[key] = file_record(Path(source_path))
    for label, source_path in config.get("legacy_reference_artifact_paths", {}).items():
        key = f"{config_name}.legacy_reference.{label}"
        source_artifact_records[key] = file_record(Path(source_path))

missing_required_source_artifacts = []
for config_name, config in notebook_configs.items():
    allow_missing_until = config.get("allow_missing_source_artifacts_until_phase")
    missing_allowed = allow_missing_until is not None and CURRENT_COMPLETED_PHASE < int(allow_missing_until)
    for label in config.get("source_artifact_paths", {}):
        key = f"{config_name}.{label}"
        if not source_artifact_records[key]["exists"] and not missing_allowed:
            missing_required_source_artifacts.append(key)
assert not missing_required_source_artifacts, (
    "Missing required source artifacts: " + ", ".join(missing_required_source_artifacts)
)

training_py_entrypoints = sorted(str(path.relative_to(REPO_ROOT)) for path in TRAINING_ROOT.glob("*.py"))
assert not training_py_entrypoints, f"Notebook-only policy violation: training/*.py files found: {training_py_entrypoints}"

model_load_check = {"status": "not_run", "reason": None, "path": "legacy/models/model_jobfit_v1.keras"}
legacy_model_path = REPO_ROOT / model_load_check["path"]
if not legacy_model_path.exists():
    model_load_check.update({"status": "failed", "reason": "legacy model artifact is missing"})
elif dependency_status.get("tensorflow"):
    try:
        import tensorflow as tf  # type: ignore

        tf.keras.models.load_model(legacy_model_path, compile=False)
    except Exception as exc:  # pragma: no cover - depends on optional TensorFlow/runtime compatibility
        model_load_check.update({"status": "failed", "reason": repr(exc)})
    else:
        model_load_check.update({"status": "passed", "reason": None})
else:
    model_load_check.update({"status": "skipped", "reason": "TensorFlow is not installed in this kernel"})

assert model_load_check["status"] != "failed", f"Model load compatibility failed: {model_load_check['reason']}"

verification_warnings = list(runtime_warnings)
if venv_hygiene["local_venv_directories_found"]:
    verification_warnings.append(
        "Ignored local notebook virtual environments are present; remove them before a clean repository handoff if desired."
    )
verification_status = "passed" if not verification_warnings else "passed_with_warnings"
verification_summary = {
    "phase_id": PHASE_ID,
    "runtime": runtime_snapshot,
    "venv_hygiene": venv_hygiene,
    "config_report": config_report,
    "required_files": file_checks,
    "planned_files": planned_file_checks,
    "source_artifacts": source_artifact_records,
    "dependency_status": dependency_status,
    "runtime_readiness": runtime_readiness,
    "training_py_entrypoints": training_py_entrypoints,
    "model_load_check": model_load_check,
    "warnings": verification_warnings,
    "status": verification_status,
}
verification_report_path = REPORTS_DIR / "phase_12_repository_hygiene_runtime_bootstrap.json"
verification_report_path.write_text(json.dumps(verification_summary, indent=2, sort_keys=True), encoding="utf-8")
file_record(verification_report_path)


{'path': 'reports/phase_12_repository_hygiene_runtime_bootstrap.json',
 'exists': True,
 'size_bytes': 15980,
 'sha256': '5ef84a21194f3d268c2dad5a70391dda9e1734a07d23ef04a96f585e24585689'}

## Step 12.5 — Notebook-only training policy

### Purpose
Document where training logic, reports, manifests, model cards, and exported artifacts live when training remains notebook-first.

### Required input
Training phase plan, report directory conventions, artifact naming conventions, and model/API ownership boundaries.

### Action
Record durable policy: training code cells live in versioned notebooks; generated reports live under `reports/`; exported model artifacts use explicit versioned paths; backend/wrapper-owned fields stay outside model-core outputs.

### Expected output
Future phases can implement training and evaluation in notebooks without creating `training/*.py` package entrypoints.

### Verification
Documentation review confirms no phase requires `python -m training.train`, `python -m training.evaluate`, or an importable training package unless explicitly approved later.


In [25]:
notebook_only_training_policy = {
    "training_execution": "Versioned .ipynb notebooks under training/notebooks/ own training execution.",
    "no_training_package_entrypoints": [
        "Do not add training/train.py for execution.",
        "Do not add training/evaluate.py for execution.",
        "Do not require python -m training.* commands for notebook phases.",
    ],
    "reports_directory": "reports/",
    "generated_report_examples": [
        "reports/phase_12_repository_hygiene_runtime_bootstrap.json",
        "reports/phase_12_notebook_configs.json",
        "reports/model_card_jobfit_v2.json",
        "reports/artifact_manifest_jobfit_v2.json",
    ],
    "export_policy": "Exported models, manifests, schemas, and model cards must use explicit versioned paths and SHA-256 hashes before promotion.",
    "model_core_outputs": ["jobFitAlignment", "atsFriendliness", "overallImpression"],
    "backend_or_wrapper_owned_outputs": ["topActionables", "sectionReviews", "job detail hydration", "auth", "persistence"],
    "phase_17_plus_embedding_contract": notebook_configs["jobfit_v2"]["embedding_contract"],
    "readiness_policy": {
        "phase_12_base": "Required stdlib runtime and repository hygiene must pass.",
        "phase_17_plus_e5": "sentence-transformers, numpy, pandas, and sklearn must be available before E5-backed training can claim readiness.",
        "tensorflow_legacy_model_load": "Optional compatibility check for legacy Keras artifact; skipped kernels must report a warning.",
    },
}

verification_summary["notebook_only_training_policy"] = notebook_only_training_policy
verification_report_path.write_text(json.dumps(verification_summary, indent=2, sort_keys=True), encoding="utf-8")

print(f"Phase 12 verification status: {verification_summary['status']}")
if verification_summary["warnings"]:
    print("Warnings:")
    for warning in verification_summary["warnings"]:
        print(f"- {warning}")
print(f"Runtime report: {verification_report_path.relative_to(REPO_ROOT)}")
print(f"Config report: {config_report_path.relative_to(REPO_ROOT)}")
notebook_only_training_policy


Phase 12 verification status: passed_with_warnings
Warnings:
- TensorFlow unavailable; legacy Keras model load compatibility is skipped in this kernel.
- Ignored local notebook virtual environments are present; remove them before a clean repository handoff if desired.
Runtime report: reports/phase_12_repository_hygiene_runtime_bootstrap.json
Config report: reports/phase_12_notebook_configs.json


{'training_execution': 'Versioned .ipynb notebooks under training/notebooks/ own training execution.',
 'no_training_package_entrypoints': ['Do not add training/train.py for execution.',
  'Do not add training/evaluate.py for execution.',
  'Do not require python -m training.* commands for notebook phases.'],
 'reports_directory': 'reports/',
 'generated_report_examples': ['reports/phase_12_repository_hygiene_runtime_bootstrap.json',
  'reports/phase_12_notebook_configs.json',
  'reports/model_card_jobfit_v2.json',
  'reports/artifact_manifest_jobfit_v2.json'],
 'export_policy': 'Exported models, manifests, schemas, and model cards must use explicit versioned paths and SHA-256 hashes before promotion.',
 'model_core_outputs': ['jobFitAlignment',
  'atsFriendliness',
  'overallImpression'],
 'backend_or_wrapper_owned_outputs': ['topActionables',
  'sectionReviews',
  'job detail hydration',
  'auth',
  'persistence'],
 'phase_17_plus_embedding_contract': {'embedding_model': 'intfloat/e5